In [1]:
import numpy as np
import pandas as pd
import os
import requests
import json
import dotenv

In [2]:
dotenv.load_dotenv()

True

In [3]:
congresskey = os.getenv('congresskey')

There are five pieces of information you need to navigate any API and its poor documentation:

1. How you get an API key (or another credential) to get access to the data (check)

2. Root: the beginning of the URL for any endpoint contained within the API

A URL: `root / endpoint ? parameters`

3. Endpoint: the middle of the URL, specifies which data store on the API to access

4. Parameters: ways to control what exactly is returned when you access an endpoint. Two kinds of parameter:

* Query parameters (appear after the ?, but requests formats this for you using the `params` argument)
* Path parameters -- supposed to be entered as part of the endpoint itself

5. How to supply your API key back to the API to prove that you are you

In [4]:
# style: I don't like putting a slash at the end of the root, because the slash is usually part of the endpoint
root = 'https://api.congress.gov/v3' 

endpoint = '/member'

params = {'format': 'json',
          'currentMember': 'true',
          'api_key': congresskey}


In [5]:
r = requests.get(root + endpoint, params=params)
r

<Response [200]>

In [6]:
r.text

'{\n    "members": [\n        {\n            "bioguideId": "W000832",\n            "depiction": {\n                "attribution": "Image courtesy of the Member",\n                "imageUrl": "https://www.congress.gov/img/member/6a986b632a68122e10181e6b_200.jpg"\n            },\n            "district": 14,\n            "name": "Wahab, Aisha",\n            "partyName": "Democratic",\n            "state": "California",\n            "terms": {\n                "item": [\n                    {\n                        "chamber": "House of Representatives",\n                        "startYear": 2026\n                    }\n                ]\n            },\n            "updateDate": "2026-09-22T07:40:25Z",\n            "url": "https://api.congress.gov/v3/member/W000832?format=json"\n        },\n        {\n            "bioguideId": "B001328",\n            "depiction": {\n                "attribution": "Image courtesy of the Member",\n                "imageUrl": "https://www.congress.gov/img/m

In [7]:
r.json()

{'members': [{'bioguideId': 'W000832',
   'depiction': {'attribution': 'Image courtesy of the Member',
    'imageUrl': 'https://www.congress.gov/img/member/6a986b632a68122e10181e6b_200.jpg'},
   'district': 14,
   'name': 'Wahab, Aisha',
   'partyName': 'Democratic',
   'state': 'California',
   'terms': {'item': [{'chamber': 'House of Representatives',
      'startYear': 2026}]},
   'updateDate': '2026-09-22T07:40:25Z',
   'url': 'https://api.congress.gov/v3/member/W000832?format=json'},
  {'bioguideId': 'B001328',
   'depiction': {'attribution': 'Image courtesy of the Member',
    'imageUrl': 'https://www.congress.gov/img/member/6a9869d92a68122e10181e62_200.jpg'},
   'district': 13,
   'name': 'Blair, Everton',
   'partyName': 'Democratic',
   'state': 'Georgia',
   'terms': {'item': [{'chamber': 'House of Representatives',
      'startYear': 2026}]},
   'updateDate': '2026-09-22T07:40:25Z',
   'url': 'https://api.congress.gov/v3/member/B001328?format=json'},
  {'bioguideId': 'G00060

1. User-Agent strings/headers

A piece of text you give to the data server to tell them a bit about who you are and what your system is

2. Navigate a JSON

3. Pagination and rate limits

In [9]:
botname = 'ds6600'
version = '0.0'
purpose = 'Class demonstration'
email = 'jkropko@virginia.edu'
useragent = f'{botname}/{version} ({purpose}; {email}) python-requests/{requests.__version__}'
headers = {'User-Agent': useragent}
headers

{'User-Agent': 'ds6600/0.0 (Class demonstration; jkropko@virginia.edu) python-requests/2.34.2'}

In [10]:
root = 'https://api.congress.gov/v3'
endpoint = '/member'
params = {'format': 'json',
          'currentMember': True,
          'api_key': congresskey}

r = requests.get(root + endpoint, params=params, headers=headers)
r

<Response [200]>

In [12]:
myjson = r.json()
myjson

{'members': [{'bioguideId': 'W000832',
   'depiction': {'attribution': 'Image courtesy of the Member',
    'imageUrl': 'https://www.congress.gov/img/member/6a986b632a68122e10181e6b_200.jpg'},
   'district': 14,
   'name': 'Wahab, Aisha',
   'partyName': 'Democratic',
   'state': 'California',
   'terms': {'item': [{'chamber': 'House of Representatives',
      'startYear': 2026}]},
   'updateDate': '2026-09-22T07:40:25Z',
   'url': 'https://api.congress.gov/v3/member/W000832?format=json'},
  {'bioguideId': 'B001328',
   'depiction': {'attribution': 'Image courtesy of the Member',
    'imageUrl': 'https://www.congress.gov/img/member/6a9869d92a68122e10181e62_200.jpg'},
   'district': 13,
   'name': 'Blair, Everton',
   'partyName': 'Democratic',
   'state': 'Georgia',
   'terms': {'item': [{'chamber': 'House of Representatives',
      'startYear': 2026}]},
   'updateDate': '2026-09-22T07:40:25Z',
   'url': 'https://api.congress.gov/v3/member/B001328?format=json'},
  {'bioguideId': 'G00060

In [20]:
myjson['members'][0]['terms']['item'][0]['chamber']

'House of Representatives'

Record path -- somewhere in the JSON is a list where each item is one row of the dataframe youn want to build. The record path is the sequence of keys to call to arrive at this list.

In this case the record path is: just members

In [21]:
congress_df = pd.json_normalize(myjson, record_path=['members'])
congress_df

,bioguideId,district,name,partyName,state,updateDate,url,depiction.attribution,depiction.imageUrl,terms.item
0,W000832,14,"Wahab, Aisha",Democratic,California,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/W000832?for...,Image courtesy of the Member,https://www.congress.gov/img/member/6a986b632a...,"[{'chamber': 'House of Representatives', 'star..."
1,B001328,13,"Blair, Everton",Democratic,Georgia,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/B001328?for...,Image courtesy of the Member,https://www.congress.gov/img/member/6a9869d92a...,"[{'chamber': 'House of Representatives', 'star..."
2,G000607,1,"Gallagher, James",Republican,California,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/G000607?for...,NaN,https://www.congress.gov/img/member/6a29d4f81c...,"[{'chamber': 'House of Representatives', 'star..."
3,M001246,11,"Mejia, Analilia",Democratic,New Jersey,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/M001246?for...,Image courtesy of the Member,https://www.congress.gov/img/member/69eb974327...,"[{'chamber': 'House of Representatives', 'star..."
4,F000485,14,"Fuller, Clay",Republican,Georgia,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/F000485?for...,Image courtesy of the Member,https://www.congress.gov/img/member/69eb95e527...,"[{'chamber': 'House of Representatives', 'star..."
5,M001245,18,"Menefee, Christian D.",Democratic,Texas,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/M001245?for...,Image courtesy of the Member,https://www.congress.gov/img/member/698206b2eb...,"[{'chamber': 'House of Representatives', 'star..."
6,V000139,7,"Van Epps, Matt",Republican,Tennessee,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/V000139?for...,Image courtesy of the Member,https://www.congress.gov/img/member/6931b7d5f4...,"[{'chamber': 'House of Representatives', 'star..."
7,G000606,7,"Grijalva, Adelita S.",Democratic,Arizona,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/G000606?for...,Image courtesy of the Member,https://www.congress.gov/img/member/6915fa2585...,"[{'chamber': 'House of Representatives', 'star..."
8,P000622,1,"Patronis, Jimmy",Republican,Florida,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/P000622?for...,Image courtesy of the Member,https://www.congress.gov/img/member/67efdb991b...,"[{'chamber': 'House of Representatives', 'star..."
9,F000484,6,"Fine, Randy",Republican,Florida,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/F000484?for...,Image courtesy of the Member,https://www.congress.gov/img/member/67efda8c1b...,"[{'chamber': 'House of Representatives', 'star..."


In [23]:
params['limit'] = '250'
#params

In [26]:
r = requests.get(root + endpoint, params=params, headers=headers)
myjson = r.json()
congress_df = pd.json_normalize(myjson, record_path=['members'])

pd.set_option('display.max_rows', 10)
congress_df

,bioguideId,district,name,partyName,state,updateDate,url,depiction.attribution,depiction.imageUrl,terms.item
0,W000832,14.0,"Wahab, Aisha",Democratic,California,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/W000832?for...,Image courtesy of the Member,https://www.congress.gov/img/member/6a986b632a...,"[{'chamber': 'House of Representatives', 'star..."
1,B001328,13.0,"Blair, Everton",Democratic,Georgia,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/B001328?for...,Image courtesy of the Member,https://www.congress.gov/img/member/6a9869d92a...,"[{'chamber': 'House of Representatives', 'star..."
2,G000607,1.0,"Gallagher, James",Republican,California,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/G000607?for...,NaN,https://www.congress.gov/img/member/6a29d4f81c...,"[{'chamber': 'House of Representatives', 'star..."
3,M001246,11.0,"Mejia, Analilia",Democratic,New Jersey,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/M001246?for...,Image courtesy of the Member,https://www.congress.gov/img/member/69eb974327...,"[{'chamber': 'House of Representatives', 'star..."
4,F000485,14.0,"Fuller, Clay",Republican,Georgia,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/F000485?for...,Image courtesy of the Member,https://www.congress.gov/img/member/69eb95e527...,"[{'chamber': 'House of Representatives', 'star..."
...,...,...,...,...,...,...,...,...,...,...
245,T000480,4.0,"Timmons, William R.",Republican,South Carolina,2026-09-22T07:40:22Z,https://api.congress.gov/v3/member/T000480?for...,Image courtesy of the Member,https://www.congress.gov/img/member/t000480_20...,"[{'chamber': 'House of Representatives', 'star..."
246,R000610,14.0,"Reschenthaler, Guy",Republican,Pennsylvania,2026-09-22T07:40:22Z,https://api.congress.gov/v3/member/R000610?for...,Image courtesy of the Member,https://www.congress.gov/img/member/r000610_20...,"[{'chamber': 'House of Representatives', 'star..."
247,J000302,13.0,"Joyce, John",Republican,Pennsylvania,2026-09-22T07:40:22Z,https://api.congress.gov/v3/member/J000302?for...,Image courtesy of the Member,https://www.congress.gov/img/member/j000302_20...,"[{'chamber': 'House of Representatives', 'star..."
248,M001204,9.0,"Meuser, Daniel",Republican,Pennsylvania,2026-09-22T07:40:22Z,https://api.congress.gov/v3/member/M001204?for...,Image courtesy of the Member,https://www.congress.gov/img/member/6776ebbf4f...,"[{'chamber': 'House of Representatives', 'star..."


In [31]:
params['offset'] = '600'
r = requests.get(root + endpoint, params=params, headers=headers)
myjson = r.json()
myjson
#congress_df = pd.json_normalize(myjson, record_path=['members'])
#congress_df

{'members': [],
 'pagination': {'count': 539,
  'prev': 'https://api.congress.gov/v3/member?currentMember=True&offset=350&limit=250&format=json'},
 'request': {'contentType': 'application/json', 'format': 'json'}}

In [34]:
i = 0
still_members = True
congress_json = []
while still_members:
    print(i)
    params['offset'] = i
    r = requests.get(root + endpoint, params=params, headers=headers)
    myjson = r.json()
    still_members = len(myjson['members']) > 0
    congress_json.extend(myjson['members'])
    i = i + 250
 




0
250
500
750


In [36]:
len(congress_json)

539

In [37]:
congress_df = pd.json_normalize(congress_json)

In [38]:
congress_df

,bioguideId,district,name,partyName,state,updateDate,url,depiction.attribution,depiction.imageUrl,terms.item
0,W000832,14.0,"Wahab, Aisha",Democratic,California,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/W000832?for...,Image courtesy of the Member,https://www.congress.gov/img/member/6a986b632a...,"[{'chamber': 'House of Representatives', 'star..."
1,B001328,13.0,"Blair, Everton",Democratic,Georgia,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/B001328?for...,Image courtesy of the Member,https://www.congress.gov/img/member/6a9869d92a...,"[{'chamber': 'House of Representatives', 'star..."
2,G000607,1.0,"Gallagher, James",Republican,California,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/G000607?for...,NaN,https://www.congress.gov/img/member/6a29d4f81c...,"[{'chamber': 'House of Representatives', 'star..."
3,M001246,11.0,"Mejia, Analilia",Democratic,New Jersey,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/M001246?for...,Image courtesy of the Member,https://www.congress.gov/img/member/69eb974327...,"[{'chamber': 'House of Representatives', 'star..."
4,F000485,14.0,"Fuller, Clay",Republican,Georgia,2026-09-22T07:40:25Z,https://api.congress.gov/v3/member/F000485?for...,Image courtesy of the Member,https://www.congress.gov/img/member/69eb95e527...,"[{'chamber': 'House of Representatives', 'star..."
...,...,...,...,...,...,...,...,...,...,...
534,B001267,NaN,"Bennet, Michael F.",Democratic,Colorado,2026-03-08T10:30:30Z,https://api.congress.gov/v3/member/B001267?for...,"<a href=""http://www.senate.gov/artandhistory/h...",https://www.congress.gov/img/member/b001267_20...,"[{'chamber': 'Senate', 'startYear': 2009}]"
535,B001261,NaN,"Barrasso, John",Republican,Wyoming,2026-03-08T10:30:29Z,https://api.congress.gov/v3/member/B001261?for...,"<a href=""http://www.senate.gov/artandhistory/h...",https://www.congress.gov/img/member/b001261_20...,"[{'chamber': 'Senate', 'startYear': 2007}]"
536,B001299,NaN,"Banks, Jim",Republican,Indiana,2026-03-08T10:30:27Z,https://api.congress.gov/v3/member/B001299?for...,Official U.S. Senate Photo,https://www.congress.gov/img/member/677d83cbfd...,"[{'chamber': 'House of Representatives', 'endY..."
537,B001230,NaN,"Baldwin, Tammy",Democratic,Wisconsin,2026-03-08T10:30:26Z,https://api.congress.gov/v3/member/B001230?for...,"<a href=""http://www.senate.gov/artandhistory/h...",https://www.congress.gov/img/member/b001230_20...,"[{'chamber': 'House of Representatives', 'endY..."
